Install libraries


In [18]:
!pip install -q transformers sentencepiece sacremoses tqdm

Check GPU

In [19]:
import torch

print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("Please enable T4 GPU")

GPU Available: True
GPU Name: Tesla T4


In [20]:
import pandas as pd

df = pd.read_csv("GWSD (1).tsv", sep="\t")

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (2300, 18)
   Unnamed: 0  round  batch sent_id  \
0           0      1      0      s0   
1           1      1      0      s1   
2           2      1      0      s2   
3           3      1      0      s3   
4           4      1      0      s4   

                                            sentence   worker_0   worker_1  \
0                          Global warming is a hoax.  disagrees  disagrees   
1  Alarming levels of sea level rise are predicte...    neutral    neutral   
2  Over the past several years, the United States...    neutral    neutral   
3  Global warming is happening and it will be dan...     agrees     agrees   
4                            Some icebergs are cute.    neutral    neutral   

    worker_2   worker_3   worker_4   worker_5   worker_6   worker_7  disagree  \
0  disagrees  disagrees  disagrees  disagrees  disagrees  disagrees       NaN   
1    neutral     agrees     agrees    neutral    neutral     agrees       NaN   
2    neutral    neutral    

Load model on GPU

In [23]:
import torch
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Using a valid multilingual translation model instead of the non-existent Helsinki-NLP one
model_name = "facebook/m2m100_418M"

tokenizer = M2M100Tokenizer.from_pretrained(model_name)
tokenizer.src_lang = "en"

model = M2M100ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32
).to(device)

model.eval()

print("Bengali-capable translation model loaded!")
print("Device:", next(model.parameters()).device)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Bengali-capable translation model loaded!
Device: cuda:0


Create translation function

In [24]:
def translate_to_bengali(texts):

    inputs = bn_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)

    with torch.inference_mode():

        outputs = bn_model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=1
        )

    return bn_tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

Translation

In [26]:
import torch
from tqdm.auto import tqdm
import os

# Redefining the function locally to use the correct active M2M100 tokenizer and model
def translate_to_bengali(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.get_lang_id("bn"),
            max_new_tokens=128,
            num_beams=1
        )

    return tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

sentences = df["sentence"].fillna("").astype(str).tolist()

bengali_sentences = []

batch_size = 32

for i in tqdm(range(0, len(sentences), batch_size)):

    batch = sentences[i:i + batch_size]

    translated = translate_to_bengali(batch)

    bengali_sentences.extend(translated)

    # Save progress after every 5 batches
    if (i // batch_size + 1) % 5 == 0:

        df["sentence_bn"] = (
            bengali_sentences +
            [""] * (len(df) - len(bengali_sentences))
        )

        df.to_csv(
            "/content/GWSD_Bengali_Progress.tsv",
            sep="\t",
            index=False,
            encoding="utf-8-sig"
        )

# Save the final results to the dataframe and output file
df["sentence_bn"] = bengali_sentences
df.to_csv(
    "/content/GWSD_Bengali_Final.tsv",
    sep="\t",
    index=False,
    encoding="utf-8-sig"
)

print("Bengali translation completed!")

  0%|          | 0/72 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Bengali translation completed!


In [28]:
from google.colab import files

df["sentence_bn"] = bengali_sentences

df.to_csv(
    "/content/GWSD_Bengali.tsv",
    sep="\t",
    index=False,
    encoding="utf-8-sig"
)

files.download("/content/GWSD_Bengali.tsv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>